<a href="https://colab.research.google.com/github/ryeggouti/morocco-world-cup-2022/blob/main/morocco_wc2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lions in the Data
**How Morocco reached the 2022 World Cup semi-final, told through numbers.**

Morocco became the first African team to reach a World Cup semi-final. This notebook uses StatsBomb's free event data (every pass and shot of all 64 matches) to find out how, and tests the story against all 32 teams.

**Website:** https://ryeggouti.github.io/morocco-world-cup-2022/ · Part of [Morocco in Data](https://ryeggouti.github.io/)

## Results at a glance
| Finding | Evidence |
|---|---|
| Fewer shots, better chances | Spain and Portugal out-shot Morocco (12 vs 6, 12 vs 9), but Morocco had the higher xG in both (0.57 vs 0.49, 0.97 vs 0.74). Per shot, Morocco's chances against Spain were worth 2.3× Spain's. |
| Defending the box | Spain and Portugal took 24 shots combined and scored 0; mostly low-quality shots, even inside the box. |
| Beating the odds | En-Nesyri scored 2 from 1.05 xG; Ziyech scored from 10 shots worth 0.31 xG. Team: 6 goals from 5.54 xG. |
| Stronger when it got harder | Shots against Morocco were worth 0.129 xG each in the group stage (28th of 32), and about 0.08 in the knockouts. Over 7 matches (0.095), 5th of the 8 teams that played 5+ matches. |

## Key idea: expected goals (xG)
Every shot gets an **xG** value from 0 to 1: how likely a shot from that position and situation is to score, based on many past shots. A team's total xG measures the quality of its chances, not just how many it took.

## 1. Setup and Morocco's seven matches

In [ ]:
!pip install statsbombpy mplsoccer -q
import warnings
warnings.filterwarnings("ignore")   # statsbombpy warns that we use free data without a login
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsbombpy import sb
from mplsoccer import VerticalPitch

RED, GREY = "#C1272D", "#999999"

comps = sb.competitions()
wc = comps[(comps.competition_name == "FIFA World Cup")
           & (comps.season_name == "2022")]
cid = int(wc.competition_id.iloc[0])
sid = int(wc.season_id.iloc[0])

matches = sb.matches(competition_id=cid, season_id=sid)
is_mar = (matches.home_team == "Morocco") | (matches.away_team == "Morocco")
mar = matches[is_mar].sort_values("match_date").reset_index(drop=True)
mar["opponent"] = np.where(mar.home_team == "Morocco", mar.away_team, mar.home_team)
mar["stage"] = ["Group", "Group", "Group", "Round of 16",
                "Quarter-final", "Semi-final", "Third place"]
print(mar[["match_date", "stage", "home_team", "away_team",
           "home_score", "away_score"]])

## 2. Load every event
`sb.events` downloads every event of a match (passes, tackles, shots...). Shots in **period 5** are penalty-shootout kicks; they're left out of shot and xG totals, because a shootout isn't open play. That's also why the data shows Morocco vs Spain as 0–0: shootout goals don't count in the score.

In [ ]:
all_events = {}
for mid in matches.match_id:
    all_events[mid] = sb.events(match_id=mid)

ev = pd.concat(all_events.values())
all_shots = ev[(ev.type == "Shot") & (ev.period < 5)]
shootouts = ev[(ev.type == "Shot") & (ev.period == 5)]
shots = all_shots[all_shots.match_id.isin(mar.match_id)]
print(len(all_events), "matches,", len(all_shots), "shots;",
      len(shots), "in Morocco's matches")

## Finding 1: fewer shots, better chances
Shots per match first, then xG, to compare quantity with quality.

In [ ]:
rows = []
for m in mar.itertuples():
    s = shots[shots.match_id == m.match_id]
    mor_s, opp_s = s[s.team == "Morocco"], s[s.team != "Morocco"]
    rows.append({"opponent": m.opponent, "stage": m.stage,
                 "morocco_shots": len(mor_s), "opponent_shots": len(opp_s),
                 "morocco_xg": mor_s.shot_statsbomb_xg.sum(),
                 "opponent_xg": opp_s.shot_statsbomb_xg.sum()})
per_match = pd.DataFrame(rows)
print(per_match.round(2).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(per_match))
for a, kind, label in [(ax[0], "shots", "Shots"), (ax[1], "xg", "Expected goals (xG)")]:
    a.bar(x - 0.2, per_match["morocco_" + kind], 0.4, color=RED, label="Morocco")
    a.bar(x + 0.2, per_match["opponent_" + kind], 0.4, color=GREY, label="Opponent")
    a.set_xticks(x, per_match.opponent, rotation=30)
    a.set_ylabel(label)
    a.legend()
ax[0].set_title("Shots per match")
ax[1].set_title("Chance quality (xG) per match")
plt.tight_layout()
plt.show()

spain = per_match[per_match.opponent == "Spain"].iloc[0]
ratio = (spain.morocco_xg / spain.morocco_shots) / (spain.opponent_xg / spain.opponent_shots)
print(f"Against Spain, Morocco's xG per shot was {ratio:.1f}x Spain's")

**Result:** Spain (12 vs 6) and Portugal (12 vs 9) took more shots, but Morocco created the higher xG in both matches (0.57 vs 0.49 and 0.97 vs 0.74). Against Spain, each Moroccan shot was worth about 2.3× as much as each Spanish one.

## Finding 2: defending the box
Every shot the four knockout opponents took. Dot size = xG, red = goal.

In [ ]:
ko_ids = mar[mar.stage != "Group"].match_id
opp_ko = shots[shots.match_id.isin(ko_ids) & (shots.team != "Morocco")]
goals = opp_ko[opp_ko.shot_outcome == "Goal"]
miss = opp_ko[opp_ko.shot_outcome != "Goal"]

pitch = VerticalPitch(half=True, pitch_type="statsbomb")
fig, ax = pitch.draw(figsize=(7, 6))
for d, color, label in [(miss, GREY, "No goal"), (goals, RED, "Goal")]:
    pitch.scatter(d.location.str[0], d.location.str[1],
                  s=d.shot_statsbomb_xg * 800 + 30, c=color,
                  edgecolors="black", ax=ax, label=label)
ax.legend(loc="lower left")
ax.set_title("Knockout opponents' shots against Morocco (bigger = higher xG)")
plt.show()

sp = per_match[per_match.opponent.isin(["Spain", "Portugal"])]
print("Spain + Portugal:", sp.opponent_shots.sum(), "shots,",
      int(opp_ko[opp_ko.team.isin(["Spain", "Portugal"])].shot_outcome.eq("Goal").sum()), "goals")

**Result:** knockout opponents often got into the box, but mostly for small, low-xG shots: from tight angles, under pressure or through crowds of defenders. Spain and Portugal took 24 shots combined and scored none. All four knockout goals conceded came against France and Croatia.

## Finding 3: beating the odds
Which Moroccan players got the best chances, and who scored more than expected.

In [ ]:
mor = shots[shots.team == "Morocco"]
players = mor.groupby("player").agg(
    shots=("id", "count"),
    xg=("shot_statsbomb_xg", "sum"),
    goals=("shot_outcome", lambda s: (s == "Goal").sum()),
).sort_values("xg", ascending=False)
print(players.head(8).round(2))
print(f"Team: {int(players.goals.sum())} goals from {players.xg.sum():.2f} xG")

top = players.head(8).iloc[::-1]
y = np.arange(len(top))
plt.figure(figsize=(8, 4.5))
plt.barh(y + 0.2, top.xg, 0.4, color=GREY, label="Expected goals (xG)")
plt.barh(y - 0.2, top.goals, 0.4, color=RED, label="Actual goals")
plt.yticks(y, top.index)
plt.legend()
plt.title("Expected vs actual goals, Morocco's top 8 by xG")
plt.tight_layout()
plt.show()

print("Every goal in the data:")
print(mor[mor.shot_outcome == "Goal"][["match_id", "player", "minute"]])

**Result:** En-Nesyri scored 2 goals from 1.05 xG, and Ziyech scored from 10 shots worth only 0.31 xG. As a team, Morocco scored 6 goals from 5.54 xG, slightly better than expected.

**Data check:** the goal list has exactly 6 goals, matching the official total. StatsBomb credits the 72' goal against Belgium to Romain Saïss, while official records credit Abdelhamid Sabiri; the team total is unaffected.

## Finding 4: stronger when it got harder
Is Morocco's defending unusual? For every team: the average xG of the shots it **conceded** (lower = opponents got worse chances). Because teams played 3 to 7 matches against different opponents, there are two fair comparisons:
1. **Group stage only:** every team played exactly 3 matches.
2. **Teams with 5+ matches:** Morocco against the other teams that went deep.

In [ ]:
def conceded_quality(match_table):
    rows = []
    teams = pd.unique(match_table[["home_team", "away_team"]].values.ravel())
    for t in teams:
        mine = match_table[(match_table.home_team == t) | (match_table.away_team == t)]
        against = all_shots[all_shots.match_id.isin(mine.match_id) & (all_shots.team != t)]
        rows.append({"team": t, "matches": len(mine),
                     "shots_against_per_match": len(against) / len(mine),
                     "xg_per_shot_against": against.shot_statsbomb_xg.mean()})
    out = pd.DataFrame(rows).sort_values("xg_per_shot_against").reset_index(drop=True)
    out["rank"] = out.index + 1
    return out

overall = conceded_quality(matches)
group = conceded_quality(matches[matches.competition_stage == "Group Stage"])
deep = overall[overall.matches >= 5].reset_index(drop=True)
deep["rank"] = deep.index + 1

print("Group stage only:")
print(group[group.team == "Morocco"].round(3).to_string(index=False))
print("\nTeams with 5+ matches:")
print(deep[["rank", "team", "matches", "xg_per_shot_against"]].round(3).to_string(index=False))

k = per_match[per_match.stage != "Group"]
print(f"\nMorocco in the knockouts: {k.opponent_xg.sum():.2f} xG from "
      f"{k.opponent_shots.sum()} shots = {k.opponent_xg.sum() / k.opponent_shots.sum():.3f} per shot")

plt.figure(figsize=(8, 6))
others = overall[overall.team != "Morocco"]
me = overall[overall.team == "Morocco"]
plt.scatter(others.shots_against_per_match, others.xg_per_shot_against, color=GREY)
plt.scatter(me.shots_against_per_match, me.xg_per_shot_against, color=RED, s=150, label="Morocco")
for r in overall.itertuples():
    plt.annotate(r.team, (r.shots_against_per_match, r.xg_per_shot_against),
                 fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.xlabel("Shots conceded per match")
plt.ylabel("xG per shot conceded (lower = worse chances for opponents)")
plt.title("All 32 teams: how dangerous were the shots they allowed?")
plt.legend()
plt.tight_layout()
plt.show()

**Result:** in the group stage, shots against Morocco were worth 0.129 xG each, 28th of 32 teams. Against Spain, Portugal, France and Croatia, that fell to about 0.08 (4.08 xG from 50 shots). Over all 7 matches (0.095), Morocco ranked 5th of the 8 teams that played 5+ matches, almost level with Argentina (0.094) and England (0.093).

*Caveat:* the group-stage figure rests on only 20 shots, and teams faced very different opponents, so the ranking is approximate. The first version of this comparison also ranked teams that played only 3 matches above Morocco; comparing teams with the same number of matches is fairer.

## 5. Export for the website
The site reads every number from two files, so nothing is typed in by hand:
- `data.json`: per-match results (including shootouts, calculated from period 5) and the top 8 players
- `shots.json`: every opponent shot in Morocco's matches, for the interactive shot map

In [ ]:
from google.colab import files

matches_out = []
for m, row in zip(mar.itertuples(), per_match.itertuples()):
    home = m.home_team == "Morocco"
    so = shootouts[shootouts.match_id == m.match_id]
    scored = so[so.shot_outcome == "Goal"]
    mor_pens = int((scored.team == "Morocco").sum())
    opp_pens = int((scored.team != "Morocco").sum())
    matches_out.append({
        "date": str(m.match_date), "stage": m.stage, "opponent": m.opponent,
        "morocco_goals": int(m.home_score if home else m.away_score),
        "opponent_goals": int(m.away_score if home else m.home_score),
        "morocco_pens": mor_pens if len(so) else None,
        "opponent_pens": opp_pens if len(so) else None,
        "morocco_shots": int(row.morocco_shots), "opponent_shots": int(row.opponent_shots),
        "morocco_xg": round(float(row.morocco_xg), 2), "opponent_xg": round(float(row.opponent_xg), 2),
    })

players_out = [{"player": name, "shots": int(r.shots), "xg": round(float(r.xg), 2), "goals": int(r.goals)}
               for name, r in players.head(8).iterrows()]

stage_of = dict(zip(mar.match_id, mar.stage))
shots_out = []
for s in shots[shots.team != "Morocco"].itertuples():
    shots_out.append({"match_id": int(s.match_id), "team": s.team,
                      "stage": stage_of[s.match_id], "player": s.player,
                      "minute": int(s.minute), "xg": round(float(s.shot_statsbomb_xg), 3),
                      "outcome": s.shot_outcome,
                      "x": float(s.location[0]), "y": float(s.location[1])})

with open("data.json", "w") as f:
    json.dump({"matches": matches_out, "players": players_out}, f, indent=2, ensure_ascii=False)
with open("shots.json", "w") as f:
    json.dump(shots_out, f, ensure_ascii=False)
print(matches_out[3])
print(len(shots_out), "opponent shots exported")
files.download("data.json")
files.download("shots.json")

## Data notes
- Data: [StatsBomb Open Data](https://github.com/statsbomb/open-data), free event data for the 2022 World Cup.
- Penalty-shootout kicks (period 5) are excluded from shot and xG totals; shootout scores are calculated from them separately.
- xG values are StatsBomb's own model.
- Comparisons between teams are approximate: teams played 3 to 7 matches against different opponents.
- I used AI tools to help write some of the code. I chose the questions, checked the results and wrote the findings.